# RLS Tiny VLM — Colab runner
This notebook only clones the repo and runs its scripts (all code lives in `src/`).
Runtime → Change runtime type → **T4 GPU**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')  # checkpoints and results go here: Colab runtimes are deleted

In [ ]:
REPO = 'https://github.com/<your-user>/rls-tiny-vlm.git'  # <- your repository
!git clone $REPO /content/rls-tiny-vlm
%cd /content/rls-tiny-vlm
# Colab already ships a CUDA build of torch: do NOT `pip install -r requirements.txt` (it would replace it)
!pip install -q pyyaml tqdm matplotlib pytest

In [ ]:
!nvidia-smi
!lscpu | grep 'Model name'
!python -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
!python generate_data.py   # ~15 s; writes data/*.pt (fixed seed)

In [ ]:
!python -m pytest -q

In [ ]:
RUNS = '/content/drive/MyDrive/rls_runs'
# E0 sanity check
!python -m experiments.E0_overfit.overfit_one_batch --device cuda

In [ ]:
SEED = 0
!python -m src.train --config configs/baseline.yaml --seed $SEED --device cuda --out-dir $RUNS/baseline_seed$SEED --resume

In [ ]:
!python -m src.evaluate --checkpoint $RUNS/baseline_seed$SEED/best.pt --split test --device cuda

In [ ]:
# Same for the blind baseline
!python -m src.train --config configs/blind.yaml --seed $SEED --device cuda --out-dir $RUNS/blind_seed$SEED --resume
!python -m src.evaluate --checkpoint $RUNS/blind_seed$SEED/best.pt --split test --device cuda